<a href="https://colab.research.google.com/github/BhagyashreeMohalkar/3D_RNA_Structure_prediction/blob/main/03_graph_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RNA3D-Predict — Graph Preprocessing

This notebook converts the curated RNA structure dataset into machine-learning-ready graph representations for RNA 3D structure prediction.

## Objective

The train, validation, and test metadata produced by `02_dataset_split.ipynb` identify the RNA structures that will be used for model development.

This notebook converts each RNA structure into a graph representation where:

- **Node:** one RNA nucleotide
- **Node features:** nucleotide identity (A, U, G, C)
- **Edges:** sequential backbone connectivity between neighboring nucleotides
- **Target:** experimental C4′ 3D coordinates for each nucleotide

## Representation

For an RNA sequence

A–U–G–C–G

the graph is represented as:

A — U — G — C — G

Each nucleotide is represented by a feature vector, while the experimental C4′ coordinates provide the ground-truth 3D target.

## Output

This notebook will produce graph-ready samples that can be consumed by the baseline, GNN, and EGNN models.

The preprocessing pipeline is designed to keep the input features separate from the experimental target coordinates to prevent target leakage.

In [1]:
# ---------------------------------------------------------
# CELL 1 — Project setup and load V1 dataset splits
# ---------------------------------------------------------
# Purpose:
#   Mount Google Drive, define the project paths, and load
#   the train/validation/test metadata created by Notebook 2.
#
# Important:
#   The CSV files tell us which RNA structures belong to each
#   split. The original PDB archive will later provide the
#   actual experimental C4' coordinates.
# ---------------------------------------------------------

from google.colab import drive
from pathlib import Path

import pandas as pd
import numpy as np


# ---------------------------------------------------------
# Mount Google Drive
# ---------------------------------------------------------

drive.mount("/content/drive")


# ---------------------------------------------------------
# Define project directories
# ---------------------------------------------------------

PROJECT_DIR = Path(
    "/content/drive/MyDrive/RNA 3D Structure Prediction"
)

DATASET_DIR = PROJECT_DIR / "Dataset"

SPLITS_DIR = DATASET_DIR / "splits"

# Raw RNAsolo archive containing the PDB structures.
RAW_DATASET_DIR = DATASET_DIR


# ---------------------------------------------------------
# Define paths to the three V1 split files
# ---------------------------------------------------------

TRAIN_PATH = SPLITS_DIR / "train_v1.csv"
VALIDATION_PATH = SPLITS_DIR / "validation_v1.csv"
TEST_PATH = SPLITS_DIR / "test_v1.csv"


# ---------------------------------------------------------
# Validate that the split files exist
# ---------------------------------------------------------

assert TRAIN_PATH.exists(), (
    f"Training split not found:\n{TRAIN_PATH}"
)

assert VALIDATION_PATH.exists(), (
    f"Validation split not found:\n{VALIDATION_PATH}"
)

assert TEST_PATH.exists(), (
    f"Test split not found:\n{TEST_PATH}"
)


# ---------------------------------------------------------
# Load the split metadata
# ---------------------------------------------------------

train_df = pd.read_csv(TRAIN_PATH)
validation_df = pd.read_csv(VALIDATION_PATH)
test_df = pd.read_csv(TEST_PATH)


# ---------------------------------------------------------
# Display basic information
# ---------------------------------------------------------

print("Train:", train_df.shape)
print("Validation:", validation_df.shape)
print("Test:", test_df.shape)

print("\nTrain columns:")
print(train_df.columns.tolist())

print("\nFirst 3 training samples:")
display(train_df.head(3))

Mounted at /content/drive
Train: (426, 19)
Validation: (55, 19)
Test: (55, 19)

Train columns:
['sample_id', 'file_name', 'pdb_id', 'model', 'filename_chains', 'pdb_chains', 'num_chains', 'num_residues', 'sequence', 'unique_residue_names', 'nonstandard_residues', 'num_nonstandard_types', 'atom_count', 'P_present', 'C4prime_present', 'P_coverage', 'C4prime_coverage', 'exact_sequence_group', 'split']

First 3 training samples:


,sample_id,file_name,pdb_id,model,filename_chains,pdb_chains,num_chains,num_residues,sequence,unique_residue_names,nonstandard_residues,num_nonstandard_types,atom_count,P_present,C4prime_present,P_coverage,C4prime_coverage,exact_sequence_group,split
0,RNA_0001,PDB_00006WLQ_1_A.pdb,6WLQ,1,A,A,1,118,GUCAUGAGUGCCAGCGUCAAGCCCCGGCUUGCUGGCCGGCAACCCU...,"A,C,G,U",NaN,0,3124,118,118,1.00000,1.0,SEQ_0446,train
1,RNA_0002,PDB_00002KX8_1_A.pdb,2KX8,1,A,A,1,42,GGGAAAUGAGGCGCUGCAUGUGGCAGUCUGCCUUUCUUUCCC,"A,C,G,U",NaN,0,1338,41,42,0.97619,1.0,SEQ_0353,train
2,RNA_0004,PDB_00004PQV_1_A.pdb,4PQV,1,A,A,1,68,GGGUCAGAUCGGCGAAAGUCGCCACUUCGCCGAGGAGUGCAAUCUG...,"A,C,G,U",NaN,0,1466,68,68,1.00000,1.0,SEQ_0390,train


In [2]:
# ---------------------------------------------------------
# CELL 2 — Locate and validate the raw PDB archive
# ---------------------------------------------------------
# Purpose:
#   Locate the RNAsolo ZIP archive containing the original
#   PDB structures and confirm that it is available.
#
# Notebook 3 will use this archive only to retrieve the
# experimental structural information needed to build graph
# targets, especially the C4' coordinates.
# ---------------------------------------------------------

import zipfile


# ---------------------------------------------------------
# Locate ZIP files inside the Dataset directory.
# ---------------------------------------------------------

zip_files = list(
    DATASET_DIR.glob("*.zip")
)

print("ZIP files found:")

for zip_file in zip_files:
    print(zip_file.name)


# ---------------------------------------------------------
# We expect exactly one raw RNAsolo ZIP archive.
# ---------------------------------------------------------

assert len(zip_files) == 1, (
    f"Expected exactly one raw ZIP archive, "
    f"but found {len(zip_files)}."
)

RAW_ZIP_PATH = zip_files[0]

print("\nRaw PDB archive:")
print(RAW_ZIP_PATH)

print(
    "\nArchive exists:",
    RAW_ZIP_PATH.exists()
)


# ---------------------------------------------------------
# Validate that the archive can be opened and contains
# PDB files.
# ---------------------------------------------------------

with zipfile.ZipFile(
    RAW_ZIP_PATH,
    "r"
) as z:

    archive_files = z.namelist()

pdb_files = [
    file_name
    for file_name in archive_files
    if file_name.lower().endswith(".pdb")
]

print(
    "\nPDB structures in archive:",
    len(pdb_files)
)

assert len(pdb_files) == 1188, (
    f"Expected 1188 PDB structures, "
    f"but found {len(pdb_files)}."
)

print("\nRaw archive validation: PASSED")

ZIP files found:
BGSU__R__All__A__1000_0__pdb_4_53.zip

Raw PDB archive:
/content/drive/MyDrive/RNA 3D Structure Prediction/Dataset/BGSU__R__All__A__1000_0__pdb_4_53.zip

Archive exists: True

PDB structures in archive: 1188

Raw archive validation: PASSED


In [3]:
# ---------------------------------------------------------
# CELL 3 — Extract one sample's sequence and C4' coordinates
# ---------------------------------------------------------
# Purpose:
#   Test the structural extraction pipeline on a single RNA
#   before processing the complete train/validation/test set.
#
# For the selected sample we extract:
#   - nucleotide sequence
#   - residue identifiers
#   - C4' 3D coordinates
#
# The sequence and coordinates must have exactly the same
# number of entries and remain in the same nucleotide order.
# ---------------------------------------------------------

import re


# ---------------------------------------------------------
# Select one known training sample.
# ---------------------------------------------------------

sample = train_df.iloc[0]

sample_id = sample["sample_id"]
file_name = sample["file_name"]
expected_sequence = sample["sequence"]

print("Sample ID:", sample_id)
print("PDB file:", file_name)
print("Expected sequence length:", len(expected_sequence))


# ---------------------------------------------------------
# Read the selected PDB file from the raw ZIP archive.
# ---------------------------------------------------------

with zipfile.ZipFile(
    RAW_ZIP_PATH,
    "r"
) as z:

    with z.open(file_name) as f:

        lines = [
            line.decode(
                "utf-8",
                errors="replace"
            ).rstrip("\n")
            for line in f
        ]


# ---------------------------------------------------------
# Extract residues and C4' coordinates from the PDB.
# ---------------------------------------------------------
# We only use ATOM records belonging to the RNA chain
# specified in the split metadata.
# ---------------------------------------------------------

target_chain = sample["pdb_chains"]

# The curated V1 dataset contains one chain per sample,
# so the chain field should contain exactly one chain ID.
assert "," not in target_chain, (
    f"Expected a single chain, but found: {target_chain}"
)

target_chain = target_chain.strip()


residues = {}
c4prime_coordinates = []


for line in lines:

    # Ignore non-coordinate records.
    if not line.startswith("ATOM"):
        continue

    # Read the chain ID from the PDB fixed-width field.
    chain_id = line[21].strip()

    # Ignore atoms belonging to another chain.
    if chain_id != target_chain:
        continue

    # Extract atom and residue information.
    atom_name = line[12:16].strip()
    residue_name = line[17:20].strip()

    residue_number = line[22:26].strip()
    insertion_code = line[26].strip()

    # We only need C4' for the coarse-grained target.
    if atom_name not in {"C4'", "C4*"}:
        continue

    # Extract Cartesian coordinates.
    x = float(line[30:38])
    y = float(line[38:46])
    z = float(line[46:54])

    residue_key = (
        residue_number,
        insertion_code
    )

    # Store one C4' coordinate per residue.
    residues[residue_key] = {
        "residue_name": residue_name,
        "coordinate": (x, y, z)
    }


# ---------------------------------------------------------
# Sort residues into sequence order.
# ---------------------------------------------------------

ordered_residues = sorted(
    residues.items(),
    key=lambda item: (
        int(item[0][0])
        if item[0][0].isdigit()
        else 0,
        item[0][1]
    )
)


# ---------------------------------------------------------
# Build the extracted sequence and coordinate array.
# ---------------------------------------------------------

extracted_sequence = "".join(
    residue["residue_name"]
    for _, residue in ordered_residues
)

extracted_coordinates = np.array(
    [
        residue["coordinate"]
        for _, residue in ordered_residues
    ],
    dtype=np.float32
)


# ---------------------------------------------------------
# Validate the extracted structure.
# ---------------------------------------------------------

print("\nExtracted sequence length:",
      len(extracted_sequence))

print(
    "Extracted coordinate shape:",
    extracted_coordinates.shape
)

print(
    "Expected coordinate shape:",
    (len(expected_sequence), 3)
)


# Verify that the sequence exactly matches the metadata.
assert extracted_sequence == expected_sequence, (
    "Extracted PDB sequence does not match the sequence "
    "stored in the split metadata."
)

# Verify one 3D coordinate per nucleotide.
assert extracted_coordinates.shape == (
    len(expected_sequence),
    3
), (
    "Number of C4' coordinates does not match "
    "the sequence length."
)

print("\nSequence match: PASSED")
print("Coordinate count: PASSED")

print("\nFirst 5 nucleotides:")
print(extracted_sequence[:5])

print("\nFirst 5 C4' coordinates:")
print(extracted_coordinates[:5])

Sample ID: RNA_0001
PDB file: PDB_00006WLQ_1_A.pdb
Expected sequence length: 118

Extracted sequence length: 118
Extracted coordinate shape: (118, 3)
Expected coordinate shape: (118, 3)

Sequence match: PASSED
Coordinate count: PASSED

First 5 nucleotides:
GUCAU

First 5 C4' coordinates:
[[ 71.595  91.574  92.125]
 [ 75.136  96.304  88.6  ]
 [ 78.166  97.412  83.342]
 [ 79.438  97.529  77.154]
 [ 82.42  101.164  78.12 ]]


In [4]:
# ---------------------------------------------------------
# CELL 4 — Create a reusable RNA structure extractor
# ---------------------------------------------------------
# Purpose:
#   Convert one curated PDB structure into:
#
#       sequence
#       C4' coordinates
#
#   while preserving the nucleotide order.
#
# This is the reusable version of the logic we validated
# in Cell 3.
# ---------------------------------------------------------

def extract_rna_structure(
    zip_file,
    file_name,
    expected_sequence,
    chain_id
):
    """
    Extract one RNA sequence and its C4' coordinates.

    Parameters
    ----------
    zip_file : zipfile.ZipFile
        Open RNAsolo ZIP archive.

    file_name : str
        PDB filename inside the archive.

    expected_sequence : str
        Sequence stored in the curated metadata.

    chain_id : str
        RNA chain to extract.

    Returns
    -------
    dict
        Extracted sequence and C4' coordinates.
    """

    # -----------------------------------------------------
    # Read the PDB file from the ZIP archive.
    # -----------------------------------------------------

    with zip_file.open(file_name) as f:

        lines = [
            line.decode(
                "utf-8",
                errors="replace"
            ).rstrip("\n")
            for line in f
        ]


    # -----------------------------------------------------
    # Store one C4' coordinate per residue.
    # -----------------------------------------------------

    residues = {}


    # -----------------------------------------------------
    # Parse ATOM records.
    # -----------------------------------------------------

    for line in lines:

        # Ignore non-coordinate records.
        if not line.startswith("ATOM"):
            continue

        # Read chain identifier.
        current_chain = line[21].strip()

        # Ignore other chains.
        if current_chain != chain_id:
            continue

        # Read atom and residue information.
        atom_name = line[12:16].strip()
        residue_name = line[17:20].strip()

        residue_number = line[22:26].strip()
        insertion_code = line[26].strip()

        # We only need the C4' atom.
        if atom_name not in {"C4'", "C4*"}:
            continue

        # Read Cartesian coordinates.
        x = float(line[30:38])
        y = float(line[38:46])
        z = float(line[46:54])

        residue_key = (
            residue_number,
            insertion_code
        )

        residues[residue_key] = {
            "residue_name": residue_name,
            "coordinate": (x, y, z)
        }


    # -----------------------------------------------------
    # Sort residues into sequence order.
    # -----------------------------------------------------

    ordered_residues = sorted(
        residues.items(),
        key=lambda item: (
            int(item[0][0])
            if item[0][0].isdigit()
            else 0,
            item[0][1]
        )
    )


    # -----------------------------------------------------
    # Build the extracted sequence.
    # -----------------------------------------------------

    extracted_sequence = "".join(
        residue["residue_name"]
        for _, residue in ordered_residues
    )


    # -----------------------------------------------------
    # Build the C4' coordinate matrix.
    #
    # Shape:
    #     (number_of_nucleotides, 3)
    # -----------------------------------------------------

    coordinates = np.array(
        [
            residue["coordinate"]
            for _, residue in ordered_residues
        ],
        dtype=np.float32
    )


    # -----------------------------------------------------
    # Validate against the metadata.
    # -----------------------------------------------------

    if extracted_sequence != expected_sequence:

        raise ValueError(
            f"Sequence mismatch for {file_name}: "
            f"extracted sequence differs from metadata."
        )


    expected_shape = (
        len(expected_sequence),
        3
    )

    if coordinates.shape != expected_shape:

        raise ValueError(
            f"Coordinate shape mismatch for {file_name}: "
            f"expected {expected_shape}, "
            f"got {coordinates.shape}."
        )


    # -----------------------------------------------------
    # Return the validated structure representation.
    # -----------------------------------------------------

    return {
        "sequence": extracted_sequence,
        "coordinates": coordinates
    }

In [5]:
# ---------------------------------------------------------
# CELL 5 — Validate the reusable structure extractor
# ---------------------------------------------------------
# Purpose:
#   Confirm that the reusable function produces exactly the
#   same sequence and coordinate matrix as our validated
#   single-sample extraction.
# ---------------------------------------------------------

# Select the first training sample.
sample = train_df.iloc[0]

# Extract the single chain ID.
chain_id = sample["pdb_chains"]

# The V1 dataset contains only single-chain samples.
assert "," not in chain_id, (
    f"Unexpected multiple chains: {chain_id}"
)

# Open the raw archive and test the reusable extractor.
with zipfile.ZipFile(
    RAW_ZIP_PATH,
    "r"
) as z:

    extracted = extract_rna_structure(
        zip_file=z,
        file_name=sample["file_name"],
        expected_sequence=sample["sequence"],
        chain_id=chain_id
    )


# ---------------------------------------------------------
# Display validation results.
# ---------------------------------------------------------

print("Sample ID:", sample["sample_id"])

print(
    "Sequence length:",
    len(extracted["sequence"])
)

print(
    "Coordinate shape:",
    extracted["coordinates"].shape
)

print(
    "Sequence match:",
    extracted["sequence"] == sample["sequence"]
)

print("\nFirst 5 coordinates:")
print(
    extracted["coordinates"][:5]
)

print("\nReusable extractor validation: PASSED")

Sample ID: RNA_0001
Sequence length: 118
Coordinate shape: (118, 3)
Sequence match: True

First 5 coordinates:
[[ 71.595  91.574  92.125]
 [ 75.136  96.304  88.6  ]
 [ 78.166  97.412  83.342]
 [ 79.438  97.529  77.154]
 [ 82.42  101.164  78.12 ]]

Reusable extractor validation: PASSED


In [6]:
# ---------------------------------------------------------
# CELL 6 — Define the V1 RNA graph representation
# ---------------------------------------------------------
# Purpose:
#   Convert one RNA sequence into the graph structure that
#   will be used by our baseline, GNN, and EGNN models.
#
# V1 representation:
#
#   Node:
#       one nucleotide
#
#   Node features:
#       one-hot encoding of A/U/G/C
#
#   Edges:
#       sequential backbone connections
#       (i <-> i+1)
#
#   Target:
#       experimental C4' coordinates
#
# Important:
#   Experimental coordinates are TARGET values only.
#   They are NOT included in the input node features.
# ---------------------------------------------------------

# Mapping from nucleotide identity to one-hot feature vector.
NUCLEOTIDE_TO_FEATURE = {
    "A": [1.0, 0.0, 0.0, 0.0],
    "U": [0.0, 1.0, 0.0, 0.0],
    "G": [0.0, 0.0, 1.0, 0.0],
    "C": [0.0, 0.0, 0.0, 1.0]
}


def build_rna_graph(sequence, coordinates):
    """
    Build a V1 graph representation for one RNA.

    Parameters
    ----------
    sequence : str
        RNA nucleotide sequence.

    coordinates : np.ndarray
        Experimental C4' coordinates with shape (N, 3).

    Returns
    -------
    dict
        Graph representation containing:
            - node_features
            - edge_index
            - coordinates
            - num_nodes
    """

    # -----------------------------------------------------
    # Basic validation
    # -----------------------------------------------------

    num_nodes = len(sequence)

    if coordinates.shape != (num_nodes, 3):
        raise ValueError(
            f"Expected coordinates with shape "
            f"({num_nodes}, 3), "
            f"got {coordinates.shape}."
        )

    # Make sure all nucleotides are valid V1 bases.
    invalid_bases = set(sequence) - set(
        NUCLEOTIDE_TO_FEATURE.keys()
    )

    if invalid_bases:
        raise ValueError(
            f"Invalid nucleotides found: {invalid_bases}"
        )


    # -----------------------------------------------------
    # Create node features
    # -----------------------------------------------------
    # One node per nucleotide.
    #
    # Shape:
    #     (N, 4)
    #
    # where the 4 columns correspond to:
    #     A, U, G, C
    # -----------------------------------------------------

    node_features = np.array(
        [
            NUCLEOTIDE_TO_FEATURE[nucleotide]
            for nucleotide in sequence
        ],
        dtype=np.float32
    )


    # -----------------------------------------------------
    # Create backbone edges
    # -----------------------------------------------------
    # Each nucleotide is connected to its immediate
    # sequence neighbor.
    #
    # For N nucleotides:
    #
    #     0 <-> 1
    #     1 <-> 2
    #     ...
    #     N-2 <-> N-1
    #
    # We store both directions because GNN message passing
    # normally uses directed edge pairs.
    # -----------------------------------------------------

    edges = []

    for i in range(num_nodes - 1):

        # Forward edge.
        edges.append(
            [i, i + 1]
        )

        # Reverse edge.
        edges.append(
            [i + 1, i]
        )

    edge_index = np.array(
        edges,
        dtype=np.int64
    ).T

    # For a single-nucleotide RNA there are no edges.
    # Make sure the resulting shape is still (2, 0).
    if num_nodes == 1:
        edge_index = np.empty(
            (2, 0),
            dtype=np.int64
        )


    # -----------------------------------------------------
    # Return the graph representation
    # -----------------------------------------------------

    return {
        "node_features": node_features,
        "edge_index": edge_index,
        "coordinates": coordinates,
        "num_nodes": num_nodes
    }

In [7]:
# ---------------------------------------------------------
# CELL 7 — Validate the V1 graph representation
# ---------------------------------------------------------
# Purpose:
#   Confirm that the graph construction produces the expected
#   number of nodes, node features, edges, and target
#   coordinates for a real RNA structure.
# ---------------------------------------------------------

# Build a graph from the structure extracted in Cell 5.
graph = build_rna_graph(
    sequence=extracted["sequence"],
    coordinates=extracted["coordinates"]
)


# ---------------------------------------------------------
# Display the graph dimensions
# ---------------------------------------------------------

print("Number of nodes:",
      graph["num_nodes"])

print(
    "Node feature shape:",
    graph["node_features"].shape
)

print(
    "Edge index shape:",
    graph["edge_index"].shape
)

print(
    "Target coordinate shape:",
    graph["coordinates"].shape
)


# ---------------------------------------------------------
# Inspect a small portion of the graph
# ---------------------------------------------------------

print("\nFirst 5 node features:")
print(
    graph["node_features"][:5]
)

print("\nFirst 10 directed edges:")
print(
    graph["edge_index"][:, :10]
)


# ---------------------------------------------------------
# Validate expected dimensions
# ---------------------------------------------------------

N = len(extracted["sequence"])

assert graph["node_features"].shape == (
    N,
    4
)

assert graph["coordinates"].shape == (
    N,
    3
)

# Every connection is represented in both directions,
# so an N-node chain has 2(N-1) directed edges.
expected_edges = (
    2 * (N - 1)
)

assert graph["edge_index"].shape == (
    2,
    expected_edges
)

print("\nGraph representation validation: PASSED")

Number of nodes: 118
Node feature shape: (118, 4)
Edge index shape: (2, 234)
Target coordinate shape: (118, 3)

First 5 node features:
[[0. 0. 1. 0.]
 [0. 1. 0. 0.]
 [0. 0. 0. 1.]
 [1. 0. 0. 0.]
 [0. 1. 0. 0.]]

First 10 directed edges:
[[0 1 1 2 2 3 3 4 4 5]
 [1 0 2 1 3 2 4 3 5 4]]

Graph representation validation: PASSED


In [8]:
# ---------------------------------------------------------
# CELL 8 — Install and import PyTorch Geometric
# ---------------------------------------------------------
# Purpose:
#   Install PyTorch Geometric and import the graph data
#   structure that will be used by our GNN/EGNN models.
#
# We are only setting up the graph-learning environment here.
# No model is trained in this cell.
# ---------------------------------------------------------

# Install PyTorch Geometric.
# Colab generally provides PyTorch already, so we only need
# the PyG package itself for our current use case.
!pip -q install torch-geometric

# Import PyTorch.
import torch

# Import the PyTorch Geometric Data object.
from torch_geometric.data import Data

# Print versions so the environment is reproducible.
print("PyTorch version:",
      torch.__version__)

print(
    "PyTorch Geometric version:",
    __import__("torch_geometric").__version__
)

print("\nPyTorch Geometric setup: PASSED")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 27.2 MB/s eta 0:00:00
PyTorch version: 2.11.0+cpu
PyTorch Geometric version: 2.8.0.post1

PyTorch Geometric setup: PASSED


In [9]:
# ---------------------------------------------------------
# CELL 9 — Convert one RNA graph into a PyG Data object
# ---------------------------------------------------------
# Purpose:
#   Convert our NumPy-based graph representation into the
#   standard PyTorch Geometric format.
#
# PyG representation:
#
#   x          → node features
#   edge_index → graph connectivity
#   y          → target C4' coordinates
#
# Experimental coordinates are stored as TARGET values only.
# ---------------------------------------------------------

# ---------------------------------------------------------
# Convert the graph components to PyTorch tensors.
# ---------------------------------------------------------

x = torch.tensor(
    graph["node_features"],
    dtype=torch.float32
)

edge_index = torch.tensor(
    graph["edge_index"],
    dtype=torch.long
)

y = torch.tensor(
    graph["coordinates"],
    dtype=torch.float32
)


# ---------------------------------------------------------
# Create the PyTorch Geometric Data object.
# ---------------------------------------------------------

data = Data(
    x=x,
    edge_index=edge_index,
    y=y
)


# ---------------------------------------------------------
# Display the resulting graph.
# ---------------------------------------------------------

print(data)

print("\nNode feature shape:",
      data.x.shape)

print(
    "Edge index shape:",
    data.edge_index.shape
)

print(
    "Target coordinate shape:",
    data.y.shape
)


# ---------------------------------------------------------
# Validate the graph object.
# ---------------------------------------------------------

assert data.x.shape == (
    graph["num_nodes"],
    4
)

assert data.edge_index.shape == (
    2,
    2 * (graph["num_nodes"] - 1)
)

assert data.y.shape == (
    graph["num_nodes"],
    3
)

print("\nPyG Data object validation: PASSED")

Data(x=[118, 4], edge_index=[2, 234], y=[118, 3])

Node feature shape: torch.Size([118, 4])
Edge index shape: torch.Size([2, 234])
Target coordinate shape: torch.Size([118, 3])

PyG Data object validation: PASSED


In [10]:
# ---------------------------------------------------------
# CELL 10 — Create a reusable RNA-to-PyG preprocessing function
# ---------------------------------------------------------
# Purpose:
#   Combine the two operations we have already validated:
#
#   1. Extract the RNA sequence and experimental C4' coordinates
#      from its PDB structure.
#
#   2. Convert the RNA sequence into our V1 graph representation.
#
# The result is a PyTorch Geometric Data object containing:
#   x          -> nucleotide node features
#   edge_index -> backbone connectivity
#   y          -> experimental C4' coordinates
#
# Metadata such as sample_id and pdb_id is also attached so
# that we can trace every graph back to its original structure.
# ---------------------------------------------------------

def preprocess_rna_to_pyg(
    zip_file,
    sample_row
):
    """
    Convert one split-metadata row into a PyG Data object.

    Parameters
    ----------
    zip_file : zipfile.ZipFile
        Open RNAsolo PDB archive.

    sample_row : pandas.Series
        One row from train_df, validation_df, or test_df.

    Returns
    -------
    Data
        PyTorch Geometric graph for one RNA.
    """

    # -----------------------------------------------------
    # Extract sample information from the metadata.
    # -----------------------------------------------------

    sample_id = sample_row["sample_id"]
    file_name = sample_row["file_name"]
    expected_sequence = sample_row["sequence"]

    # V1 contains only single-chain structures.
    chain_id = sample_row["pdb_chains"]

    if "," in chain_id:
        raise ValueError(
            f"{sample_id} contains multiple chains: {chain_id}"
        )


    # -----------------------------------------------------
    # Extract sequence + experimental C4' coordinates.
    # -----------------------------------------------------

    structure = extract_rna_structure(
        zip_file=zip_file,
        file_name=file_name,
        expected_sequence=expected_sequence,
        chain_id=chain_id
    )

    sequence = structure["sequence"]
    coordinates = structure["coordinates"]


    # -----------------------------------------------------
    # Build our V1 RNA graph.
    # -----------------------------------------------------

    graph = build_rna_graph(
        sequence=sequence,
        coordinates=coordinates
    )


    # -----------------------------------------------------
    # Convert graph arrays to PyTorch tensors.
    # -----------------------------------------------------

    x = torch.tensor(
        graph["node_features"],
        dtype=torch.float32
    )

    edge_index = torch.tensor(
        graph["edge_index"],
        dtype=torch.long
    )

    y = torch.tensor(
        graph["coordinates"],
        dtype=torch.float32
    )


    # -----------------------------------------------------
    # Create the PyG Data object.
    # -----------------------------------------------------

    data = Data(
        x=x,
        edge_index=edge_index,
        y=y
    )


    # -----------------------------------------------------
    # Attach useful metadata.
    #
    # These attributes do not serve as model features.
    # They allow us to trace predictions back to the original
    # RNA structure during evaluation.
    # -----------------------------------------------------

    data.sample_id = sample_id
    data.pdb_id = str(sample_row["pdb_id"])
    data.sequence = sequence
    data.num_nodes = len(sequence)
    data.split = str(sample_row["split"])


    # -----------------------------------------------------
    # Final consistency checks.
    # -----------------------------------------------------

    if data.x.size(0) != len(sequence):
        raise ValueError(
            f"Node count mismatch for {sample_id}."
        )

    if data.y.size(0) != len(sequence):
        raise ValueError(
            f"Target coordinate count mismatch for {sample_id}."
        )

    if data.edge_index.size(1) != (
        2 * (len(sequence) - 1)
    ):
        raise ValueError(
            f"Unexpected edge count for {sample_id}."
        )


    return data

In [11]:
# ---------------------------------------------------------
# CELL 11 — Validate the complete preprocessing pipeline
# ---------------------------------------------------------
# Purpose:
#   Run the complete PDB → graph pipeline on one training
#   sample and verify that the resulting PyG Data object
#   contains the expected features, edges, targets, and
#   metadata.
# ---------------------------------------------------------

# Select the first training sample.
sample = train_df.iloc[0]


# ---------------------------------------------------------
# Open the raw PDB archive and preprocess the sample.
# ---------------------------------------------------------

with zipfile.ZipFile(
    RAW_ZIP_PATH,
    "r"
) as z:

    sample_graph = preprocess_rna_to_pyg(
        zip_file=z,
        sample_row=sample
    )


# ---------------------------------------------------------
# Display the resulting graph.
# ---------------------------------------------------------

print(sample_graph)

print("\nSample ID:",
      sample_graph.sample_id)

print("PDB ID:",
      sample_graph.pdb_id)

print("Sequence length:",
      sample_graph.num_nodes)

print("Split:",
      sample_graph.split)

print("\nNode features:",
      sample_graph.x.shape)

print("Edges:",
      sample_graph.edge_index.shape)

print("Target coordinates:",
      sample_graph.y.shape)


# ---------------------------------------------------------
# Validate the complete object.
# ---------------------------------------------------------

assert (
    sample_graph.x.shape[0]
    == sample["num_residues"]
)

assert (
    sample_graph.x.shape[1]
    == 4
)

assert (
    sample_graph.y.shape
    == (
        sample["num_residues"],
        3
    )
)

assert (
    sample_graph.sample_id
    == sample["sample_id"]
)

print("\nComplete preprocessing pipeline: PASSED")

Data(x=[118, 4], edge_index=[2, 234], y=[118, 3], sample_id='RNA_0001', pdb_id='6WLQ', sequence='GUCAUGAGUGCCAGCGUCAAGCCCCGGCUUGCUGGCCGGCAACCCUCCAACCGCGGUGGGGUGCCCCGGGUGAUGACCAGGUUGAGUAGCCGUGACGGCUACGCGGCAAGCGCGGGUC', num_nodes=118, split='train')

Sample ID: RNA_0001
PDB ID: 6WLQ
Sequence length: 118
Split: train

Node features: torch.Size([118, 4])
Edges: torch.Size([2, 234])
Target coordinates: torch.Size([118, 3])

Complete preprocessing pipeline: PASSED


In [12]:
# ---------------------------------------------------------
# CELL 12 — Process the complete V1 dataset
# ---------------------------------------------------------
# Purpose:
#   Convert every structure in the train, validation, and
#   test splits into a validated PyTorch Geometric graph.
#
# Expected dataset:
#   Train      → 426 graphs
#   Validation → 55 graphs
#   Test       → 55 graphs
#
# Total:
#   536 graphs
#
# Any parsing/preprocessing failure is recorded explicitly
# instead of silently being skipped.
# ---------------------------------------------------------


def process_split_dataframe(
    dataframe,
    split_name,
    zip_file
):
    """
    Process every RNA structure in one dataset split.

    Parameters
    ----------
    dataframe : pandas.DataFrame
        Metadata for one split.

    split_name : str
        Name of the split: train, validation, or test.

    zip_file : zipfile.ZipFile
        Open PDB archive.

    Returns
    -------
    graphs : list
        Successfully processed PyG graphs.

    failures : list
        Information about any failed structures.
    """

    graphs = []
    failures = []

    total = len(dataframe)

    print(
        f"\nProcessing {split_name}: "
        f"{total} structures"
    )

    for position, (_, row) in enumerate(
        dataframe.iterrows(),
        start=1
    ):

        try:

            # Convert the current structure into a PyG graph.
            graph = preprocess_rna_to_pyg(
                zip_file=zip_file,
                sample_row=row
            )

            graphs.append(graph)

        except Exception as error:

            # Record the failure so we can investigate it later.
            failures.append({
                "sample_id": row["sample_id"],
                "file_name": row["file_name"],
                "error": str(error)
            })

        # Print progress every 50 structures and at the end.
        if (
            position % 50 == 0
            or position == total
        ):

            print(
                f"Processed {position}/{total}"
            )

    return graphs, failures


# ---------------------------------------------------------
# Open the raw PDB archive once and process all splits.
# ---------------------------------------------------------

with zipfile.ZipFile(
    RAW_ZIP_PATH,
    "r"
) as z:

    train_graphs, train_failures = (
        process_split_dataframe(
            train_df,
            "train",
            z
        )
    )

    validation_graphs, validation_failures = (
        process_split_dataframe(
            validation_df,
            "validation",
            z
        )
    )

    test_graphs, test_failures = (
        process_split_dataframe(
            test_df,
            "test",
            z
        )
    )


# ---------------------------------------------------------
# Report the final preprocessing results.
# ---------------------------------------------------------

print("\n==============================================")
print("PREPROCESSING SUMMARY")
print("==============================================")

print(
    "Train graphs:",
    len(train_graphs),
    "/",
    len(train_df)
)

print(
    "Validation graphs:",
    len(validation_graphs),
    "/",
    len(validation_df)
)

print(
    "Test graphs:",
    len(test_graphs),
    "/",
    len(test_df)
)

total_graphs = (
    len(train_graphs)
    + len(validation_graphs)
    + len(test_graphs)
)

total_failures = (
    len(train_failures)
    + len(validation_failures)
    + len(test_failures)
)

print(
    "\nTotal successful graphs:",
    total_graphs
)

print(
    "Total failures:",
    total_failures
)


Processing train: 426 structures
Processed 50/426
Processed 100/426
Processed 150/426
Processed 200/426
Processed 250/426
Processed 300/426
Processed 350/426
Processed 400/426
Processed 426/426

Processing validation: 55 structures
Processed 50/55
Processed 55/55

Processing test: 55 structures
Processed 50/55
Processed 55/55

PREPROCESSING SUMMARY
Train graphs: 426 / 426
Validation graphs: 55 / 55
Test graphs: 55 / 55

Total successful graphs: 536
Total failures: 0


In [13]:
# ---------------------------------------------------------
# CELL 13 — Validate all processed graph objects
# ---------------------------------------------------------
# Purpose:
#   Perform a final dataset-wide quality check on the 536
#   PyTorch Geometric graphs created in Cell 12.
#
# Checks:
#   1. Expected number of graphs in each split
#   2. Node feature shape = (N, 4)
#   3. Target coordinate shape = (N, 3)
#   4. Edge count = 2(N-1)
#   5. No NaN/Inf values in features or targets
#   6. Metadata is present and consistent
#
# No graphs are modified in this cell.
# ---------------------------------------------------------

def validate_graph_list(graphs, expected_count, split_name):
    """
    Validate every graph in one split.
    """

    print(f"\nValidating {split_name}...")

    # -----------------------------------------------------
    # Check the number of graphs.
    # -----------------------------------------------------

    assert len(graphs) == expected_count, (
        f"{split_name}: expected {expected_count} graphs, "
        f"found {len(graphs)}."
    )

    for graph in graphs:

        # -------------------------------------------------
        # Number of nucleotide nodes.
        # -------------------------------------------------

        num_nodes = graph.x.size(0)

        # -------------------------------------------------
        # Node feature dimensions.
        # -------------------------------------------------

        assert graph.x.shape == (
            num_nodes,
            4
        ), (
            f"{graph.sample_id}: invalid node feature shape "
            f"{graph.x.shape}"
        )

        # -------------------------------------------------
        # Target coordinate dimensions.
        # -------------------------------------------------

        assert graph.y.shape == (
            num_nodes,
            3
        ), (
            f"{graph.sample_id}: invalid target shape "
            f"{graph.y.shape}"
        )

        # -------------------------------------------------
        # Backbone edge count.
        # -------------------------------------------------

        expected_edges = (
            2 * (num_nodes - 1)
        )

        assert graph.edge_index.shape == (
            2,
            expected_edges
        ), (
            f"{graph.sample_id}: invalid edge shape "
            f"{graph.edge_index.shape}"
        )

        # -------------------------------------------------
        # Check for NaN or infinite values.
        # -------------------------------------------------

        assert torch.isfinite(
            graph.x
        ).all(), (
            f"{graph.sample_id}: invalid node features."
        )

        assert torch.isfinite(
            graph.y
        ).all(), (
            f"{graph.sample_id}: invalid coordinates."
        )

        # -------------------------------------------------
        # Metadata checks.
        # -------------------------------------------------

        assert hasattr(
            graph,
            "sample_id"
        )

        assert hasattr(
            graph,
            "pdb_id"
        )

        assert hasattr(
            graph,
            "sequence"
        )

        assert graph.num_nodes == len(
            graph.sequence
        ), (
            f"{graph.sample_id}: sequence/node mismatch."
        )

    print(
        f"{split_name}: validation PASSED "
        f"({len(graphs)} graphs)"
    )


# ---------------------------------------------------------
# Validate each split.
# ---------------------------------------------------------

validate_graph_list(
    train_graphs,
    expected_count=426,
    split_name="Train"
)

validate_graph_list(
    validation_graphs,
    expected_count=55,
    split_name="Validation"
)

validate_graph_list(
    test_graphs,
    expected_count=55,
    split_name="Test"
)

print("\n==============================================")
print("COMPLETE GRAPH DATASET VALIDATION: PASSED")
print("==============================================")


Validating Train...
Train: validation PASSED (426 graphs)

Validating Validation...
Validation: validation PASSED (55 graphs)

Validating Test...
Test: validation PASSED (55 graphs)

COMPLETE GRAPH DATASET VALIDATION: PASSED


In [14]:
# ---------------------------------------------------------
# CELL 14 — Save the processed PyG graph datasets
# ---------------------------------------------------------
# Purpose:
#   Save the validated train, validation, and test graph
#   objects to Google Drive.
#
# These files become the direct input to the model-training
# notebooks.
#
# We keep the three splits separate to preserve the leakage-
# controlled dataset organization created in Notebook 2.
# ---------------------------------------------------------

from pathlib import Path
import torch
import json


# ---------------------------------------------------------
# Create the processed-data directory.
# ---------------------------------------------------------

PROCESSED_DIR = (
    DATASET_DIR / "processed"
)

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ---------------------------------------------------------
# Define output paths.
# ---------------------------------------------------------

TRAIN_GRAPH_PATH = (
    PROCESSED_DIR / "train_v1.pt"
)

VALIDATION_GRAPH_PATH = (
    PROCESSED_DIR / "validation_v1.pt"
)

TEST_GRAPH_PATH = (
    PROCESSED_DIR / "test_v1.pt"
)


# ---------------------------------------------------------
# Save each graph list.
#
# PyTorch serialization preserves:
#   - node features
#   - edge_index
#   - target coordinates
#   - sample metadata
# ---------------------------------------------------------

torch.save(
    train_graphs,
    TRAIN_GRAPH_PATH
)

torch.save(
    validation_graphs,
    VALIDATION_GRAPH_PATH
)

torch.save(
    test_graphs,
    TEST_GRAPH_PATH
)


# ---------------------------------------------------------
# Report the saved files.
# ---------------------------------------------------------

print("Processed graph datasets saved:\n")

print("Train:")
print(TRAIN_GRAPH_PATH)

print("\nValidation:")
print(VALIDATION_GRAPH_PATH)

print("\nTest:")
print(TEST_GRAPH_PATH)

Processed graph datasets saved:

Train:
/content/drive/MyDrive/RNA 3D Structure Prediction/Dataset/processed/train_v1.pt

Validation:
/content/drive/MyDrive/RNA 3D Structure Prediction/Dataset/processed/validation_v1.pt

Test:
/content/drive/MyDrive/RNA 3D Structure Prediction/Dataset/processed/test_v1.pt


In [15]:
# ---------------------------------------------------------
# CELL 15 — Create the processed dataset manifest
# ---------------------------------------------------------
# Purpose:
#   Record important information about the processed V1
#   dataset so that future notebooks can understand exactly
#   what the saved graph files contain.
# ---------------------------------------------------------

manifest = {
    "dataset_name": "RNA3D-Predict",
    "dataset_version": "V1",
    "source": "RNAsolo2 representative PDB structures",

    "curation": {
        "single_chain": True,
        "min_length": 20,
        "max_length": 200,
        "standard_bases_only": True,
        "target_atom": "C4'",
        "complete_c4prime_required": True
    },

    "graph": {
        "node": "RNA nucleotide",
        "node_features": "one-hot A/U/G/C",
        "edge_type": "bidirectional sequential backbone",
        "target": "experimental C4' coordinates"
    },

    "splits": {
        "train": len(train_graphs),
        "validation": len(validation_graphs),
        "test": len(test_graphs)
    },

    "total_graphs": (
        len(train_graphs)
        + len(validation_graphs)
        + len(test_graphs)
    )
}

MANIFEST_PATH = (
    PROCESSED_DIR / "manifest_v1.json"
)

with open(
    MANIFEST_PATH,
    "w"
) as f:

    json.dump(
        manifest,
        f,
        indent=2
    )

print("Manifest saved to:")
print(MANIFEST_PATH)

print("\nManifest contents:")
print(
    json.dumps(
        manifest,
        indent=2
    )
)

Manifest saved to:
/content/drive/MyDrive/RNA 3D Structure Prediction/Dataset/processed/manifest_v1.json

Manifest contents:
{
  "dataset_name": "RNA3D-Predict",
  "dataset_version": "V1",
  "source": "RNAsolo2 representative PDB structures",
  "curation": {
    "single_chain": true,
    "min_length": 20,
    "max_length": 200,
    "standard_bases_only": true,
    "target_atom": "C4'",
    "complete_c4prime_required": true
  },
  "graph": {
    "node": "RNA nucleotide",
    "node_features": "one-hot A/U/G/C",
    "edge_type": "bidirectional sequential backbone",
    "target": "experimental C4' coordinates"
  },
  "splits": {
    "train": 426,
    "validation": 55,
    "test": 55
  },
  "total_graphs": 536
}


In [16]:
# ---------------------------------------------------------
# CELL 16 — Reload and verify saved graph datasets
# ---------------------------------------------------------
# Purpose:
#   Confirm that the graph files saved in Cell 14 can be
#   reloaded successfully and still contain the expected
#   number of valid PyG graph objects.
# ---------------------------------------------------------

# Reload the saved graph datasets.
reloaded_train = torch.load(
    TRAIN_GRAPH_PATH,
    weights_only=False
)

reloaded_validation = torch.load(
    VALIDATION_GRAPH_PATH,
    weights_only=False
)

reloaded_test = torch.load(
    TEST_GRAPH_PATH,
    weights_only=False
)


# ---------------------------------------------------------
# Verify row counts.
# ---------------------------------------------------------

print("Reloaded train graphs:",
      len(reloaded_train))

print("Reloaded validation graphs:",
      len(reloaded_validation))

print("Reloaded test graphs:",
      len(reloaded_test))


# ---------------------------------------------------------
# Verify the expected counts.
# ---------------------------------------------------------

assert len(reloaded_train) == 426
assert len(reloaded_validation) == 55
assert len(reloaded_test) == 55


# ---------------------------------------------------------
# Verify one graph after reloading.
# ---------------------------------------------------------

example = reloaded_train[0]

print("\nExample reloaded graph:")
print(example)

print("\nSample ID:",
      example.sample_id)

print("PDB ID:",
      example.pdb_id)

print("Node features:",
      example.x.shape)

print("Edges:",
      example.edge_index.shape)

print("Target:",
      example.y.shape)


print("\nSaved graph verification: PASSED")

Reloaded train graphs: 426
Reloaded validation graphs: 55
Reloaded test graphs: 55

Example reloaded graph:
Data(x=[118, 4], edge_index=[2, 234], y=[118, 3], sample_id='RNA_0001', pdb_id='6WLQ', sequence='GUCAUGAGUGCCAGCGUCAAGCCCCGGCUUGCUGGCCGGCAACCCUCCAACCGCGGUGGGGUGCCCCGGGUGAUGACCAGGUUGAGUAGCCGUGACGGCUACGCGGCAAGCGCGGGUC', num_nodes=118, split='train')

Sample ID: RNA_0001
PDB ID: 6WLQ
Node features: torch.Size([118, 4])
Edges: torch.Size([2, 234])
Target: torch.Size([118, 3])

Saved graph verification: PASSED


# Graph Preprocessing — Conclusion

The curated RNA dataset has been successfully converted into machine-learning-ready graph representations.

## Final V1 Graph Representation

For each RNA:

- **Node:** one nucleotide
- **Node features:** one-hot encoding of A, U, G, and C
- **Edges:** bidirectional sequential backbone connections between neighboring nucleotides
- **Target:** experimental C4′ 3D coordinates `(x, y, z)`

## Processed Dataset

All 536 curated structures were successfully processed:

- **Training:** 426 graphs
- **Validation:** 55 graphs
- **Test:** 55 graphs
- **Processing failures:** 0

Every graph was validated for:

- Correct node-feature dimensions
- Correct backbone connectivity
- One C4′ coordinate per nucleotide
- Absence of invalid numerical values
- Consistency between sequence length and graph dimensions
- Preservation of sample and PDB metadata

The processed graph datasets were saved as:

- `train_v1.pt`
- `validation_v1.pt`
- `test_v1.pt`

A dataset manifest was also created to document the preprocessing configuration.

## Next Stage

The graph dataset is now ready for machine-learning experiments.

The next notebook will establish a **sequence-based baseline model** before introducing the geometry-aware EGNN model. This will allow us to measure whether explicitly modeling RNA geometry provides a meaningful improvement in 3D structure prediction.